# Mrigi 25 — Inspect filtered_patents_on_zeolite.json

Quickly inspect file size, format (array vs JSON Lines), record count, and a simple field-type schema.

Target: `/home/jupyter/Mrigi/Zeolite_RAG/filtered_patents_on_zeolite.json`

In [1]:
from pathlib import Path
import os, json
from collections import defaultdict, Counter
from typing import Any, Dict, Iterable, List, Tuple
import pandas as pd

FILE_PATH = Path("/home/jupyter/Mrigi/Zeolite_RAG/filtered_patents_on_zeolite.json")
assert FILE_PATH.exists(), f"File not found: {FILE_PATH}"
print(f"Target file: {FILE_PATH}")

Target file: /home/jupyter/Mrigi/Zeolite_RAG/filtered_patents_on_zeolite.json


In [ ]:
# Basic file stats and preview
size_bytes = FILE_PATH.stat().st_size
size_mb = size_bytes / (1024*1024)
print(f"Size: {size_mb:.2f} MB ({size_bytes:,} bytes)")

with FILE_PATH.open('r', encoding='utf-8', errors='replace') as f:
    head_lines = [f.readline() for _ in range(20)]
head = ''.join(head_lines)
print("\nFirst ~20 lines preview:\n" + head[:1500])

SyntaxError: EOL while scanning string literal (3646227162.py, line 9)

In [ ]:
# Helpers to analyze JSON/JSONL
def typeof(v: Any) -> str:
    if v is None: return 'null'
    if isinstance(v, bool): return 'bool'
    if isinstance(v, int): return 'int'
    if isinstance(v, float): return 'float'
    if isinstance(v, str): return 'str'
    if isinstance(v, list): return 'list'
    if isinstance(v, dict): return 'dict'
    return type(v).__name__

def analyze_iter(objs: Iterable[Dict], limit_examples: int = 3):
    total = 0
    schema = defaultdict(Counter)
    examples: List[Dict] = []
    for obj in objs:
        if not isinstance(obj, dict):
            obj = {'_value': obj}
        total += 1
        if len(examples) < limit_examples:
            examples.append(obj)
        for k, v in obj.items():
            schema[k][typeof(v)] += 1
    return total, schema, examples

def load_any(path: Path):
    """Detect format and yield objects. Returns (format, iterator_fn)."""
    # Peek first non-space char
    with path.open('r', encoding='utf-8', errors='replace') as f:
        first_non_space = None
        while True:
            ch = f.read(1)
            if not ch: break
            if not ch.isspace():
                first_non_space = ch
                break
    if first_non_space == '[':
        fmt = 'json_array'
        def it():
            data = json.load(path.open('r', encoding='utf-8', errors='replace'))
            if isinstance(data, list):
                for item in data: yield item
            else:
                yield data
        return fmt, it
    else:
        # Try to parse as a single JSON first
        try:
            data = json.load(path.open('r', encoding='utf-8', errors='replace'))
            fmt = 'json_object' if isinstance(data, dict) else 'json_array'
            def it():
                if isinstance(data, dict):
                    if 'records' in data and isinstance(data['records'], list):
                        for item in data['records']: yield item
                    else:
                        yield data
                elif isinstance(data, list):
                    for item in data: yield item
                else:
                    yield data
            return fmt, it
        except json.JSONDecodeError:
            # Treat as JSON Lines
            fmt = 'jsonl'
            def it():
                with path.open('r', encoding='utf-8', errors='replace') as f:
                    for line in f:
                        line = line.strip()
                        if not line: continue
                        try:
                            yield json.loads(line)
                        except json.JSONDecodeError:
                            yield {'_raw': line}
            return fmt, it

In [ ]:
# Detect format and summarize schema
fmt, iterator = load_any(FILE_PATH)
print(f'Detected format: {fmt}')

total, schema, examples = analyze_iter(iterator())
print(f'Total records: {total:,}')

rows = []
for k, cnt in schema.items():
    total_key = sum(cnt.values())
    top = ', '.join(f"{t}:{n}" for t, n in cnt.most_common())
    rows.append({'field': k, 'observed_types': top, 'non_null_count': total_key, 'missing_count': total - total_key})
schema_df = pd.DataFrame(rows).sort_values(by=['missing_count', 'field']).reset_index(drop=True)
display(schema_df.head(30))

print("\nExamples (up to 3):")
for i, ex in enumerate(examples, 1):
    print(f"\nExample {i}:")
    try:
        print(json.dumps(ex, ensure_ascii=False, indent=2)[:2000])
    except Exception:
        print(str(ex)[:2000])

In [ ]:
# Optional: quick stats for common text fields if present
candidate_text_fields = ['title', 'abstract', 'claims', 'description', 'text']
present = [c for c in candidate_text_fields if c in schema]
print(f'Text-like fields present: {present}')

from typing import Iterable
def quick_len_stats(field: str, it: Iterable[dict], sample: int = 5000):
    n = 0; lengths = []
    for obj in it:
        if not isinstance(obj, dict):
            continue
        v = obj.get(field)
        if isinstance(v, str):
            lengths.append(len(v))
        n += 1
        if n >= sample:
            break
    if not lengths:
        return {'count': 0}
    s = pd.Series(lengths)
    return {'count': int(len(lengths)), 'min': int(s.min()), 'mean': float(s.mean()), 'p95': int(s.quantile(0.95)), 'max': int(s.max())}

for f in present:
    stats = quick_len_stats(f, iterator())
    print(f"{f} length stats (up to 5k records): {stats}")